# qAIR-vNext v41 -- Colab Training & Evaluation

Run cells top to bottom in order. Requires a GPU runtime
(Runtime -> Change runtime type -> GPU).

**What changed from v40:** `n_qubits` 12 -> 6 and the embedding model
`all-mpnet-base-v2` (768-dim) -> `all-MiniLM-L6-v2` (384-dim), both to
cut the quantum circuit's per-epoch simulation cost (which scales
roughly `O(n_qubits * 2^n_qubits)`). Batch size 8 -> 16 and epochs
20 -> 30 to use the compute headroom that frees up. Hypothesis
generation during cache building is now batched (`generate_batch`)
instead of one question at a time.

**Prerequisite:** the `qAIR-CSE499B` repo this notebook clones must
already have these v41 changes (`config.py`, `training/dataset.py`,
`models/generator.py`, `training/runner.py`) pushed to `main` -- this
notebook always pulls the latest `main`, not your local working copy.

**This uses a fresh Drive folder (`qAIR_V41`)** -- separate from any
`qAIR_V40` cache/checkpoints, since those were built with a different
embedding dimension (768 vs. 384) and are not compatible with this
version. `training/dataset.py` will raise a clear error rather than
silently loading a mismatched cache if you point it at the old folder.

## 1. Mount Drive & set up directories

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

BASE = "/content/drive/MyDrive/qAIR_V41"

CACHE_DIR = f"{BASE}/cache"
CKPT_DIR = f"{BASE}/ckpt"
LOG_DIR = f"{BASE}/logs"
EXPORT_DIR = f"{BASE}/exports"

for p in [CACHE_DIR, CKPT_DIR, LOG_DIR, EXPORT_DIR]:
    os.makedirs(p, exist_ok=True)

## 2. Install core deps (before cloning, so the tokenizer/model cache warms early)

In [ ]:
!pip install -q transformers==4.52.4 huggingface_hub==0.33.4

## 3. Clone the repo

In [ ]:
!rm -rf /content/qAIR-CSE499B
%cd /content
!git clone https://github.com/Thorfast191/qAIR-CSE499B.git
%cd /content/qAIR-CSE499B

## 4. Install project requirements\n\nColab ships TensorFlow/Keras preinstalled, which can shadow `transformers`' backend detection -- remove them first.

In [ ]:
!pip uninstall -y tensorflow tensorflow-cpu keras tf-keras
!pip install -r requirements.txt

## 5. Seed

In [ ]:
from training.seed import set_seed
set_seed(42)

## 6. Smoke test: generator + encoder\n\nQuick sanity check that the LLM and embedding model load and produce sane output before committing to a full ablation run. `enc.dim` should print 384 -- if it doesn't, `config.py` and this notebook have drifted.

In [ ]:
from models.generator import HypothesisGenerator
from models.encoder import HypothesisEncoder

gen = HypothesisGenerator()
hyps = gen.generate(
    "What gas do plants absorb from the atmosphere?",
    ["Oxygen", "Nitrogen", "Carbon dioxide", "Hydrogen"],
)
for i, h in enumerate(hyps):
    print(f"{i+1}. {h}")

enc = HypothesisEncoder()
emb = enc.encode(hyps)
print("embedding dim:", emb.shape[-1])

## 6b. Smoke test: batched generation\n\nThis is what `training/dataset.py` now uses to build the cache -- multiple questions generated in a single `model.generate()` call instead of one at a time. Confirms the batched path produces one hypothesis list per question, same as calling `generate()` individually.

In [ ]:
batch_hyps = gen.generate_batch(
    questions=[
        "What gas do plants absorb from the atmosphere?",
        "Why does ice float on water?",
    ],
    options_list=[
        ["Oxygen", "Nitrogen", "Carbon dioxide", "Hydrogen"],
        ["It is heavier than water", "It is less dense than water", "It absorbs heat", "It contains oxygen"],
    ],
)
for i, hyps in enumerate(batch_hyps):
    print(f"--- question {i+1} ---")
    for h in hyps:
        print(" ", h)

## 7. Helpers

In [ ]:
import torch


def checkpoint_status(path, label="Checkpoint"):
    print("=" * 60)
    print(label)
    print("=" * 60)
    print("Path:", path)
    print("Exists:", os.path.exists(path))
    if not os.path.exists(path):
        print("[NO CHECKPOINT FOUND] Will train from scratch.")
        return None
    try:
        ckpt = torch.load(path, map_location="cpu")
        epoch = ckpt.get("epoch", "UNKNOWN")
        print(f"epoch = {epoch}")
        if isinstance(epoch, int):
            print(f"start_epoch = {epoch + 1}")
        return ckpt
    except Exception as e:
        print("[CHECKPOINT INCOMPATIBLE / FAILED TO LOAD]")
        print(e)
        return None


def save_report(metrics, export_dir, filename="evaluation.txt"):
    os.makedirs(export_dir, exist_ok=True)
    report_path = os.path.join(export_dir, filename)
    with open(report_path, "w") as f:
        for k, v in metrics.items():
            f.write(f"{k}: {v}\n")
    print("Saved to:", report_path)
    return report_path


def list_exports(export_dir):
    print("=" * 60)
    print("EXPORTED FILES")
    print("=" * 60)
    for f in sorted(os.listdir(export_dir)):
        print(f)


device = "cuda" if torch.cuda.is_available() else "cpu"

## 8. Check for existing ablation checkpoints (resume support)

In [ ]:
for name in ["A1_baseline", "A1b_quantum_only", "A2_validator", "A3_persistent", "A4_full_hybrid"]:
    print()
    checkpoint_status(os.path.join(CKPT_DIR, f"{name}_latest.pt"), label=name)

## 9. Run the ablation suite\n\nBuilds/reuses the full ARC dataset cache (no `max_samples` cap -- uses the full dataset), then trains each config in `training/ablations.py::ABLATIONS`, resuming from any checkpoints found above.\n\n`n_qubits=6` and `epochs=30` reflect the v41 defaults in `config.py`; passed explicitly here so this cell stays correct even if you experiment with different local defaults.

In [ ]:
from training.ablations import run_ablation_suite, ABLATIONS

try:
    ablation_results = run_ablation_suite(
        cache_dir=CACHE_DIR,
        ckpt_dir=CKPT_DIR,
        epochs=30,
        patience=5,
        n_qubits=6,
    )
    print("\n" + "=" * 60)
    print("ABLATION RESULTS")
    print("=" * 60)
    for name, r in ablation_results.items():
        print(f"\n{name}")
        print(f"  config        : {r['config']}")
        print(f"  best_acc      : {r['best_acc']:.4f}")
        print(f"  final_val_acc : {r.get('final_val_acc')}")
except Exception as e:
    print("\n[ABLATION FAILED]")
    print(e)

## 10. Pick the best config\n\nInspect `ablation_results` above and set this to the config with the highest `best_acc`.

In [ ]:
BEST_CONFIG_NAME = "A4_full_hybrid"   # <-- change based on ablation_results above

## 11. Load the chosen checkpoint

In [ ]:
from evaluation.sample_inference import load_ablation_model, run_sample_evaluation
from training.ablations import ABLATIONS

model = load_ablation_model(
    ckpt_dir=CKPT_DIR,
    name=BEST_CONFIG_NAME,
    cfg=ABLATIONS[BEST_CONFIG_NAME],
    device=device,
    n_qubits=6,
)

## 12. Sanity-check the cache

In [ ]:
c = torch.load(f"{CACHE_DIR}/arc_train.pt", map_location="cpu")
sample = c["samples"][0] if isinstance(c, dict) else c[0]
print("H shape:", sample["H"].shape)
print("O shape:", sample["O"].shape)
print("complete:", c.get("complete") if isinstance(c, dict) else "old format")

## 13. Full validation-set evaluation\n\n`shuffle_options=False` is required here -- the default (`True`) reshuffles each sample's option order every epoch, which makes these metrics non-reproducible across runs of the same checkpoint.

In [ ]:
from functools import partial
from training.dataset import QAIRDataset, collate_fn
from torch.utils.data import DataLoader
from training.evaluate import evaluate

val_ds = QAIRDataset(split="validation", max_samples=None, cache_dir=CACHE_DIR)
val_loader = DataLoader(
    val_ds,
    batch_size=16,
    shuffle=False,
    collate_fn=partial(collate_fn, shuffle_options=False),
)

metrics = evaluate(model, val_loader, device)
print(metrics)

report_path = save_report(metrics, EXPORT_DIR)

## 14. Grab one sample for visualization\n\n`shuffle_options=True` (the default) is fine here -- this is just picking a random example to visualize, not computing aggregate metrics.

In [ ]:
sample_loader = DataLoader(val_ds, batch_size=1, shuffle=True, collate_fn=collate_fn)
sample = next(iter(sample_loader))

H = sample["H"].to(device)
O = sample["O"].to(device)

with torch.no_grad():
    out = model(H, O)

## 15. Visualizations

In [ ]:
from visualization.attention_maps import plot_attention_map

attention = out["attention"]
if attention.dim() == 4:
    attention = attention[0, 0]
elif attention.dim() == 3:
    attention = attention[0]

plot_attention_map(attention, save_path=f"{EXPORT_DIR}/attention_map.png")

In [ ]:
from visualization.energy_maps import plot_energy_map
plot_energy_map(out["answer_energy"][0], save_path=f"{EXPORT_DIR}/energy_map.png")

In [ ]:
import matplotlib.pyplot as plt

collapse = out["collapse_probs"][0].detach().cpu().numpy()
plt.figure(figsize=(6, 4))
plt.bar(range(len(collapse)), collapse)
plt.xlabel("Hypothesis")
plt.ylabel("Probability")
plt.title("Collapse Distribution")
plt.savefig(f"{EXPORT_DIR}/collapse_probs.png", bbox_inches="tight")
plt.show()

In [ ]:
from visualization.trajectory import plot_trajectory
plot_trajectory(out["trajectory"], save_path=f"{EXPORT_DIR}/trajectory.png")

In [ ]:
validator = out["validator"]
if validator is not None:
    scores = [
        validator["causal"][0].mean().item(),
        validator["diversity"][0].mean().item(),
        validator["specificity"][0].mean().item(),
        validator["relevance"][0].mean().item(),
    ]
    labels = ["causal", "diversity", "specificity", "relevance"]
    plt.figure(figsize=(7, 4))
    plt.bar(labels, scores)
    plt.title("Validator Scores")
    plt.savefig(f"{EXPORT_DIR}/validator.png", bbox_inches="tight")
    plt.show()

In [ ]:
list_exports(EXPORT_DIR)

## 16. Sample-question accuracy check

In [ ]:
sample_acc = run_sample_evaluation(model=model, device=device)
print(f"\nSample Accuracy ({BEST_CONFIG_NAME}) = {sample_acc:.4f}")

## 17. Raw output dump (debugging)

In [ ]:
print("Energy:")
print(out["answer_energy"])
print("Collapse:")
print(out["collapse_probs"])
print("Attention:")
print(out["attention"].shape)
print("Trajectory:")
print(len(out["trajectory"]))